In [ ]:
#Data Path

In [132]:
DATA_PATH = Path(r"/Users/apple/Desktop/Numpy_Ninja/Python_Session/Python_Hackathon_Sep_2026/cardiac_failure")
DATA_PATH=Path(".")
CLEANED_PATH = DATA_PATH / "cleaned_data"
CLEANED_PATH.mkdir(parents=True,exist_ok=True)

print("Raw data folder:", DATA_PATH)
print("Cleaned data folder:", CLEANED_PATH)
cardiac.to_csv(CLEANED_PATH/"cardiac_cleaned.csv",index=False)
print("saved successfully.")

Raw data folder: .
Cleaned data folder: cleaned_data
saved successfully.


In [129]:
Saved:r"/Users/apple/Desktop/Numpy_Ninja/Python_Session/Python_Hackathon_Sep_2026/cardiac_failure/cleaned_data"



In [ ]:
Loading the Dataset


In [ ]:
demography = pd.read_csv(DATA_PATH / "demography.csv")
hospitalization = pd.read_csv(DATA_PATH / "hospitalization_discharge.csv")
cardiac = pd.read_csv(DATA_PATH / "cardiac_complications.csv")
labs = pd.read_csv(DATA_PATH / "labs.csv")
patient_prescriptions = pd.read_csv(DATA_PATH / "patient_precriptions.csv")
patient_history = pd.read_csv(DATA_PATH / "patienthistory.csv")
responsiveness = pd.read_csv(DATA_PATH / "responsivenes.csv")

data_dictionary = pd.read_excel(
    DATA_PATH / "Cardiac_failure_data_dictionary.xlsx"
)

print("All datasets loaded successfully.")

In [ ]:
Understanding the dataset

In [55]:
datasets = {
    "Demography": demography,
    "Hospitalization & Discharge": hospitalization,
    "Cardiac Complications": cardiac,
    "Labs": labs,
    "Patient Prescriptions": patient_prescriptions,
    "Patient History": patient_history,
    "Responsiveness": responsiveness
}

overview = pd.DataFrame([
    {
        "Dataset": name,
        "Rows": df.shape[0],
        "Columns": df.shape[1],
        "Unique Patients": df["inpatient_number"].nunique(),
        "Duplicate Patient IDs": df["inpatient_number"].duplicated().sum(),
        "Patients with Multiple Records": df["inpatient_number"].value_counts().gt(1).sum(),
        "One Row per Patient": df.shape[0] == df["inpatient_number"].nunique(),
    }
    for name, df in datasets.items()
])

overview

,Dataset,Rows,Columns,Unique Patients,Duplicate Patient IDs,Patients with Multiple Records,One Row per Patient
0,Demography,2009,7,2009,0,0,True
1,Hospitalization & Discharge,2008,21,2008,0,0,True
2,Cardiac Complications,2008,14,2008,0,0,True
3,Labs,2008,107,2008,0,0,True
4,Patient Prescriptions,15362,2,2007,13355,1998,False
5,Patient History,2008,17,2008,0,0,True
6,Responsiveness,2008,6,2008,0,0,True


### The next table is the Cardiac Complications

### 1. Readable file

The first step is to check whether the file is readable and also make sure the contents and size of the file are matching.

In [60]:
cardiac = pd.read_csv(DATA_PATH / "cardiac_complications.csv")
print(cardiac.shape)

(2008, 14)


In [53]:
print(cardiac.head(5))

   inpatient_number  nyha_cardiac_function_classification  killip_grade  \
0            848044                                     3             1   
1            821472                                     3             1   
2            866373                                     3             2   
3            793147                                     3             3   
4            739615                                     3             2   

   myocardial_infarction  congestive_heart_failure  \
0                      0                         0   
1                      0                         1   
2                      0                         1   
3                      0                         0   
4                      0                         0   

   peripheral_vascular_disease type_of_heart_failure  lvef  \
0                            0                  Both   NaN   
1                            0                  Both   NaN   
2                            0        

In [ ]:
cardiac.describe()

In [57]:
cardiac.count()

inpatient_number                              2008
nyha_cardiac_function_classification          2008
killip_grade                                  2008
myocardial_infarction                         2008
congestive_heart_failure                      2008
peripheral_vascular_disease                   2008
type_of_heart_failure                         2008
lvef                                           635
left_ventricular_end_diastolic_diameter_lv    1311
mitral_valve_ems                               980
mitral_valve_ams                               550
ea                                             393
tricuspid_valve_return_velocity                790
tricuspid_valve_return_pressure                182
dtype: int64

### 2. Removal of Duplicate Records 

When working with medical records, we must ensure the inpatient_number is entirely unique. If a patient was admitted twice or an extraction script ran twice, it creates duplicate rows. 

In [70]:
duplicate_count = cardiac.duplicated(subset=['inpatient_number']).sum()
print(f"Number of duplicate patient records identified: {duplicate_count}")

# Remove the duplicate record, keeping the first occurrence
cardiac_cleaned = cardiac.drop_duplicates(subset=['inpatient_number'], keep='first')


Number of duplicate patient records identified: 0


### 3. Calculating the number of missing values per column 

In [76]:

missing_count = cardiac.isnull().sum()

# Calculate the percentage of missing values per column
missing_percentage = (cardiac.isnull().sum() / len(cardiac)) * 100

# Combine results into a scannable summary table
missing_summary = pd.DataFrame({
    'Missing Values Count': missing_count,
    'Percentage (%)': missing_percentage
})

# Filter to only show columns that actually contain missing values (sorted highest to lowest)
missing_summary = missing_summary[missing_summary['Missing Values Count'] > 0].sort_values(by='Missing Values Count', ascending=False)

print("--- Missing Data Summary ---")
print(missing_summary)


--- Missing Data Summary ---
                                            Missing Values Count  \
tricuspid_valve_return_pressure                             1826   
ea                                                          1615   
mitral_valve_ams                                            1458   
lvef                                                        1373   
tricuspid_valve_return_velocity                             1218   
mitral_valve_ems                                            1028   
left_ventricular_end_diastolic_diameter_lv                   697   

                                            Percentage (%)  
tricuspid_valve_return_pressure                  90.936255  
ea                                               80.428287  
mitral_valve_ams                                 72.609562  
lvef                                             68.376494  
tricuspid_valve_return_velocity                  60.657371  
mitral_valve_ems                                 51.195219  

### 4. Analysing the columns with missing values 

In the Cardiac table, the columns Ivef and ea have only few values. In ea 80% of the data is missing. Instead of removing the column we are replacing with NaN values in the data set.

### 5. Removing missing columns with empty values.

First, drop columns that are 100% empty (lvef and ea) 
If we dropped rows instead, we would end up with 0 rows of data.  Handle columns with partial missing values using Median Imputation
We use the median because it is not skewed by extreme outliner values

In [75]:
columns_with_nans = [
    'left_ventricular_end_diastolic_diameter_lv', 
    'mitral_valve_ems', 
    'mitral_valve_ams', 
    'tricuspid_valve_return_velocity', 
    'tricuspid_valve_return_pressure'
]

for col in columns_with_nans:
    # Convert column to numeric just in case there are string spaces, ignoring errors
    cardiac_cleaned[col] = pd.to_numeric(cardiac_cleaned[col], errors='coerce')
    # Fill missing values with the median of that specific column
    col_median = cardiac_cleaned[col].median()
    cardiac_cleaned[col] = cardiac_cleaned[col].fillna(col_median)

#  Final verification: Check if any missing values remain
print("--- Remaining Missing Values Check ---")
print(cardiac_cleaned.isnull().sum())
print("\n" + "="*50 + "\n")

#  View the Cleaned Dataset (First 10 rows)
pd.set_option('display.max_columns', None) # Ensures no columns are cut off in console view
print("--- Cleaned Medical Data Sample ---")
print(cardiac_cleaned.head(10))

--- Remaining Missing Values Check ---
inpatient_number                                 0
nyha_cardiac_function_classification             0
killip_grade                                     0
myocardial_infarction                            0
congestive_heart_failure                         0
peripheral_vascular_disease                      0
type_of_heart_failure                            0
lvef                                          1373
left_ventricular_end_diastolic_diameter_lv       0
mitral_valve_ems                                 0
mitral_valve_ams                                 0
ea                                            1615
tricuspid_valve_return_velocity                  0
tricuspid_valve_return_pressure                  0
dtype: int64


--- Cleaned Medical Data Sample ---
   inpatient_number  nyha_cardiac_function_classification  killip_grade  \
0            848044                                     3             1   
1            821472                           

### 6. Checking for NULL values

The Cardiac table has some NULL values in few of the columns like myocardial_infarction,congestive_heart_failure,peripheral_vascular_disease.For better analysis, we need to drop the columns that are completely empty.

In [77]:
 #Drop the 100% empty columns ('lvef' and 'ea')
cardiac_clean = cardiac.dropna(axis=1, how='all')

# 3. Fill the remaining partially blank columns with their statistical median
# This completely eliminates all remaining NaN/Null values
numeric_cols = cardiac_clean.select_dtypes(include=['float64', 'int64']).columns
for col in numeric_cols:
    cardiac_clean[col] = cardiac_clean[col].fillna(cardiac_clean[col].median())

# For object/text columns that might contain missing items, fill with most frequent value
object_cols = cardiac_clean.select_dtypes(include=['object']).columns
for col in object_cols:
    cardiac_clean[col] = cardiac_clean[col].fillna(cardiac_clean[col].mode()[0])

# 4. Save your fully clean dataset
cardiac_clean.to_csv('heart_failure_data_cleaned.csv', index=False)

# 5. Verify the null count is exactly zero across all columns
print("--- Remaining Null Values Check ---")
print(cardiac_clean.isnull().sum())

--- Remaining Null Values Check ---
inpatient_number                              0
nyha_cardiac_function_classification          0
killip_grade                                  0
myocardial_infarction                         0
congestive_heart_failure                      0
peripheral_vascular_disease                   0
type_of_heart_failure                         0
lvef                                          0
left_ventricular_end_diastolic_diameter_lv    0
mitral_valve_ems                              0
mitral_valve_ams                              0
ea                                            0
tricuspid_valve_return_velocity               0
tricuspid_valve_return_pressure               0
dtype: int64


### 7. Checking the data consistency


The data set has different datatypes -Float and Interger. Type of Heart Failure has Left,Right,Both. We should make sure the consistency of the data is correct.

In [79]:
print("=== 1. Checking Data Types and Structural Properties ===")
print(cardiac.dtypes)
print("\n" + "="*50)

print("=== 2. Auditing Categorical Text Standardisation ===")
# Ensure there are no hidden spaces or duplicate categories due to typos
print("Unique values in 'type_of_heart_failure':", cardiac['type_of_heart_failure'].unique())
print("Unique values in 'nyha_cardiac_function_classification':", cardiac['nyha_cardiac_function_classification'].unique())

print("\n" + "="*50)
print("=== 3. Spotting Numerical Consistency & Outliers ===")
# Generate descriptive stats to spot impossible min/max thresholds
echo_metrics = [
    'left_ventricular_end_diastolic_diameter_lv', 
    'mitral_valve_ems', 
    'mitral_valve_ams', 
    'tricuspid_valve_return_velocity', 
    'tricuspid_valve_return_pressure'
]

# Convert columns to numeric, forcing any error strings into NaNs for standard summary calculation
for col in echo_metrics:
    cardiac[col] = pd.to_numeric(cardiac[col], errors='coerce')

print(cardiac[echo_metrics].describe().loc[['min', 'max', 'mean']])

print("\n" + "="*50)
print("=== 4. Flagging Medical Logic Contradictions ===")
# Flag patients labeled as No Heart Attack (0) but given an Acute Heart Attack severity score (>1)
conflicting_patients = cardiac[(cardiac['myocardial_infarction'] == 0) & (cardiac['killip_grade'] > 1)]
print(f"Number of rows with MI = 0 but active Killip Grade (>1): {len(conflicting_patients)}")
if len(conflicting_patients) > 0:
    print("Sample of conflicting inpatient IDs:", conflicting_patients['inpatient_number'].head(5).tolist())

=== 1. Checking Data Types and Structural Properties ===
inpatient_number                                int64
nyha_cardiac_function_classification            int64
killip_grade                                    int64
myocardial_infarction                           int64
congestive_heart_failure                        int64
peripheral_vascular_disease                     int64
type_of_heart_failure                          object
lvef                                          float64
left_ventricular_end_diastolic_diameter_lv    float64
mitral_valve_ems                              float64
mitral_valve_ams                              float64
ea                                            float64
tricuspid_valve_return_velocity               float64
tricuspid_valve_return_pressure               float64
dtype: object

=== 2. Auditing Categorical Text Standardisation ===
Unique values in 'type_of_heart_failure': ['Both' 'Left' 'Right']
Unique values in 'nyha_cardiac_function_classificatio

In [80]:
print("--- Step 1: Fixing Decimal Outliers ---")
# Identify columns where values over 100 are physically impossible physiological traits
velocity_cols = ['mitral_valve_ems', 'mitral_valve_ams']

for col in velocity_cols:
    # Ensure column is numeric
    cardiac_clean[col] = pd.to_numeric(cardiac_clean[col], errors='coerce')
    
    # Track rows changing for confirmation
    outliers_count = (cardiac_clean[col] > 100).sum()
    if outliers_count > 0:
        print(f"Fixed {outliers_count} decimal shift outliers in '{col}'")
        
    # If a value is greater than 100, divide it by 100 to fix the missing decimal point
    cardiac_clean[col] = np.where(cardiac_clean[col] > 100, cardiac_clean[col] / 100, cardiac_clean[col])

print("\n--- Step 2: Imputing Remaining Blank Spaces (Medians) ---")
# Complete missing value treatment with the median before encoding text
numeric_cols = cardiac_clean.select_dtypes(include=['float64', 'int64']).columns
for col in numeric_cols:
    cardiac_clean[col] = cardiac_clean[col].fillna(cardiac_clean[col].median())

print("\n--- Step 3: Encoding Text Columns (Categorical Variables) ---")
print("Original unique categories:", cardiac_clean['type_of_heart_failure'].unique())

# One-Hot Encode 'type_of_heart_failure' (Creates separate 0/1 columns for Left, Right, Both)
# This is preferred over Label Encoding so algorithms don't infer an incorrect mathematical order.
cardiac_encoded = pd.get_dummies(cardiac_clean, columns=['type_of_heart_failure'], prefix='hf_type', dtype=int)

# 2. View final structure
print("\n--- Step 4: Verification Summary ---")
print(f"Final Data Shape: {df_encoded.shape} Rows and Columns")
print("\nSample of final clean, encoded columns:")
print(cardiac_encoded[['inpatient_number', 'mitral_valve_ems', 'hf_type_Both', 'hf_type_Left', 'hf_type_Right']].head())

# Save the finalized model-ready dataset
cardiac_encoded.to_csv('heart_failure_ready_for_ml.csv', index=False)

--- Step 1: Fixing Decimal Outliers ---
Fixed 9 decimal shift outliers in 'mitral_valve_ems'
Fixed 4 decimal shift outliers in 'mitral_valve_ams'

--- Step 2: Imputing Remaining Blank Spaces (Medians) ---

--- Step 3: Encoding Text Columns (Categorical Variables) ---
Original unique categories: ['Both' 'Left' 'Right']

--- Step 4: Verification Summary ---
Final Data Shape: (2008, 16) Rows and Columns

Sample of final clean, encoded columns:
   inpatient_number  mitral_valve_ems  hf_type_Both  hf_type_Left  \
0            848044              1.00             1             0   
1            821472              1.18             1             0   
2            866373              0.59             1             0   
3            793147              1.01             0             1   
4            739615              0.38             1             0   

   hf_type_Right  
0              0  
1              0  
2              0  
3              0  
4              0  


### 8. Checking the "Hidden" String Text in Numeric Columns

Columns like left_ventricular_end_diastolic_diameter_lv contain blank string text spaces instead of true numbers in rows where data is missing. If you don't explicitly force these columns to be numeric, Python reads the entire column as a text object rather than integers or floats. You must coerce these column types.

### 9. High-Risk Outlier Shifts

Beyond the Mitral Valve)While we fixed 408 and 409 in the mitral valve columns, look at row 177: left_ventricular_end_diastolic_diameter_lv has a value of 0.93.A left ventricular diameter (LVEDD) of 0.93 mm or cm is physiologically impossible for a living adult patient (normal ranges are roughly 35–60 mm).Given the column position, this looks like a data entry shift error where an ejection fraction or velocity metric was accidentally typed into the chamber size column. This row needs to be scrubbed or replaced with the column median.

### 10. Out-of-Bounds Target 

Out-of-Bounds Target Scaling ErrorLook at row 429: tricuspid_valve_return_velocity displays a value of 99. A tricuspid regurgitation velocity of 99 m/s is physically impossible (severe values rarely exceed 5 m/s). This is highly likely a placeholder value used by the data entry operator to signify "Missing/Unknown" rather than an actual clinical measurement. Leaving 99 in the data will completely destroy your model's mathematical weights.


### 11. Logical Clinical Violations


As noted in the consistency check, you have rows where myocardial_infarction = 0 (No Heart Attack), but the patient is given an acute killip_grade of 2, 3, or 4. In a rigorous medical dataset, these must be resolved by setting a rule (e.g., forcing the Killip grade to 1 or NaN if no acute heart attack event is documented).

In [81]:
#Step 1: Force numeric conversion and turn hidden string spaces into true NaNs
echo_cols = [
    'left_ventricular_end_diastolic_diameter_lv', 
    'mitral_valve_ems', 'mitral_valve_ams', 
    'tricuspid_valve_return_velocity', 'tricuspid_valve_return_pressure'
]
for col in echo_cols:
    cardiac[col] = pd.to_numeric(cardiac[col], errors='coerce')

# Step 2: Fix placeholder values and impossible physiological entries
# Fix the "99" placeholder in tricuspid velocity
cardiac['tricuspid_valve_return_velocity'] = cardiac['tricuspid_valve_return_velocity'].replace(99, np.nan)

# Fix the physically impossible 0.93 chamber diameter entry
cardiac['left_ventricular_end_diastolic_diameter_lv'] = np.where(
    cardiac['left_ventricular_end_diastolic_diameter_lv'] < 10, 
    np.nan, 
    cardiac['left_ventricular_end_diastolic_diameter_lv']
)

# Step 3: Fix the decimal shifts (e.g., 408 -> 4.08)
for col in ['mitral_valve_ems', 'mitral_valve_ams']:
    cardiac[col] = np.where(cardiac[col] > 100, cardiac[col] / 100, cardiac[col])

# Step 4: Drop completely empty columns and impute structural medians
cardiac_clean = cardiac.drop(columns=['lvef', 'ea'])
for col in cardiac_clean.select_dtypes(include=[np.number]).columns:
    cardiac_clean[col] = cardiac_clean[col].fillna(cardiac_clean[col].median())

# Step 5: Enforce clinical cross-logic consistency
# If no MI occurred, set Killip Grade to 1 (Normal/No heart failure signs post-MI)
cardiac_clean.loc[cardiac_clean['myocardial_infarction'] == 0, 'killip_grade'] = 1

print("--- Final Post-Cleaning Sanity Metrics ---")
print(cardiac_clean[echo_cols].describe().loc[['min', 'max']])

--- Final Post-Cleaning Sanity Metrics ---
     left_ventricular_end_diastolic_diameter_lv  mitral_valve_ems  \
min                                        22.0              0.03   
max                                        88.0             44.00   

     mitral_valve_ams  tricuspid_valve_return_velocity  \
min              0.06                             0.90   
max             99.00                             5.76   

     tricuspid_valve_return_pressure  
min                              1.7  
max                             87.0  


In [67]:
print(cardiac.tail())

      inpatient_number  nyha_cardiac_function_classification  killip_grade  \
2003            756045                                     4             2   
2004            726701                                     3             2   
2005            754589                                     3             2   
2006            740689                                     4             1   
2007            734280                                     4             2   

      myocardial_infarction  congestive_heart_failure  \
2003                      0                         1   
2004                      0                         1   
2005                      0                         1   
2006                      0                         1   
2007                      0                         1   

      peripheral_vascular_disease type_of_heart_failure  lvef  \
2003                            0                  Both   NaN   
2004                            0                  Both  5

In [117]:
print(cardiac["ea"].describe())

count    393.000000
mean       1.289686
std        1.289603
min        0.061600
25%        0.656000
50%        0.980000
75%        1.606000
max       21.300000
Name: ea, dtype: float64


### To verify the cleaned dataset for cardiac table


In [127]:
cardiac.to_csv(
    CLEANED_PATH / "cardiac_cleaned.csv",
    index=False
)

print("Saved:", CLEANED_PATH / "cardiac_cleaned.csv")

Saved: cleaned_data/cardiac_cleaned.csv


Saved:"/Users/apple/Desktop/Numpy_Ninja/Python_Session/Python_Hackathon_Sep_2026/cardiac_failure/cleaned_data.csv"

In [125]:
cardiac_check = pd.read_csv("cleaned_data/cardiac_cleaned.csv")
  

print("Saved shape:", cardiac_check.shape)
print("Saved columns:", cardiac_check.columns.tolist())

Saved shape: (2008, 14)
Saved columns: ['inpatient_number', 'nyha_cardiac_function_classification', 'killip_grade', 'myocardial_infarction', 'congestive_heart_failure', 'peripheral_vascular_disease', 'type_of_heart_failure', 'lvef', 'left_ventricular_end_diastolic_diameter_lv', 'mitral_valve_ems', 'mitral_valve_ams', 'ea', 'tricuspid_valve_return_velocity', 'tricuspid_valve_return_pressure']


In [101]:
dataset_overview = pd.DataFrame([
    {
        "Dataset": name,
        "Rows": df.shape[0],
        "Columns": df.shape[1],
        "Unique Patients": df["inpatient_number"].nunique(),
        "Duplicate Patient IDs": df["inpatient_number"].duplicated().sum(),
        "One Row per Patient": df.shape[0] == df["inpatient_number"].nunique()
    }
    for name, df in datasets.items()
])

dataset_overview

,Dataset,Rows,Columns,Unique Patients,Duplicate Patient IDs,One Row per Patient
0,Demography,2009,7,2009,0,True
1,Hospitalization & Discharge,2008,21,2008,0,True
2,Cardiac Complications,2008,14,2008,0,True
3,Labs,2008,107,2008,0,True
4,Patient Prescriptions,15362,2,2007,13355,False
5,Patient History,2008,17,2008,0,True
6,Responsiveness,2008,6,2008,0,True


### Patient Prescriptions Table

In [63]:
patient_prescriptions = pd.read_csv(DATA_PATH / "patient_precriptions.csv")
print(patient_prescriptions.shape)

(15362, 2)


In [65]:
patient_prescriptions.describe()

,inpatient_number
count,15362.000000
mean,798892.435295
std,41066.780871
min,722128.000000
25%,764993.000000
50%,799246.000000
75%,830868.000000
max,905720.000000


In [66]:
patient_prescriptions.count()

inpatient_number    15362
drug_name           15362
dtype: int64

### 1. Missing data/values

Checking for any missing (null) values in the dataset

Missing data can cause errors in downstream analytical pipelines. We must check both columns for null values. If any are found, we typically drop them or fill them depending on the context.



In [82]:
# Check for missing values
missing_summary = patient_prescriptions.isnull().sum()
print("Missing values per column:")
print(missing_summary)

# Action: If there were missing entries, we would remove them using:
# patient_prescriptions = patient_prescriptions.dropna()

Missing values per column:
inpatient_number    0
drug_name           0
dtype: int64


### 2. Getting Duplicate records  

In [83]:
# Count duplicates
duplicate_count = patient_prescriptions.duplicated().sum()
print(f"Number of exact duplicate rows found: {duplicate_count}")

# Remove duplicates (keeping the first occurrence)
patient_prescriptions = patient_prescriptions.drop_duplicates()
print(f"Shape after removing exact duplicates: {patient_prescriptions.shape}")

Number of exact duplicate rows found: 0
Shape after removing exact duplicates: (15362, 2)


### 3. Data Inconsistency

The patient_prescriptions has few inconssitent data and blank spaces in the prescription.

1. There are some Key Inconsistencies Found in the data.
1a. FileCase-Sensitivity Mismatches: 
The same medication is recorded under different capitalizations (e.g., metoprolol tartrate injection vs. Metoprolol Succinate Sustained-release tablet vs. Furosemide tablet). Python treats these as completely different drugs, which fractures group-by counts.

1b. Implicit Whitespace Blanks: Leading and trailing white spaces are embedded dynamically in some row entries, causing unexpected mismatches when trying to count matching strings.

1c. Trailing Commas & Trailing Rows: The file contains trailing structural symbols that must be cleanly removed so that the dataset reads purely into a relational patient-to-drug grid.

In [84]:
# For demonstration, reading the raw data context safely
raw_data = """inpatient_number,drug_name
857781,sulfotanshinone sodium injection
857781,Furosemide tablet
743087,metoprolol tartrate injection
743087,Aspirin enteric-coated tablet""" # ... (Rest of data loads dynamically)

patient_prescriptions = pd.read_csv(io.StringIO(raw_data))

print(f"Original Row Count: {len(patient_prescriptions)}")

# Step 1: Remove any pure duplicate records where a patient is assigned the exact same drug twice
patient_prescriptions_clean = patient_prescriptions.drop_duplicates()

# Step 2: Strip away invisible leading or trailing white spaces from the drug name strings
patient_prescriptions_clean['drug_name'] = patient_prescriptions_clean['drug_name'].str.strip()

# Step 3: Standardize text casing toTitle Case 
# This fixes conflicts like 'metoprolol tartrate injection' vs 'Metoprolol Succinate...'
patient_prescriptions_clean['drug_name'] = patient_prescriptions_clean['drug_name'].str.title()

# Step 4: Drop any incomplete rows where the medication field or patient number is missing
patient_prescriptions_clean = patient_prescriptions_clean.dropna(subset=['inpatient_number', 'drug_name'])

# Step 5: Enforce standard integer type properties on the ID column
patient_prescriptions_clean['inpatient_number'] = patient_prescriptions_clean['inpatient_number'].astype(int)

print(f"Cleaned Row Count: {len(patient_prescriptions_clean)}")
print("\n=== Verification Sample of the Clean Dataset ===")
print(patient_prescriptions_clean.head(10))

# Save the finalized, standardized data array
patient_prescriptions_clean.to_csv('patient_drugs_cleaned.csv', index=False)

Original Row Count: 4
Cleaned Row Count: 4

=== Verification Sample of the Clean Dataset ===
   inpatient_number                         drug_name
0            857781  Sulfotanshinone Sodium Injection
1            857781                 Furosemide Tablet
2            743087     Metoprolol Tartrate Injection
3            743087     Aspirin Enteric-Coated Tablet


In [85]:
print(patient_prescriptions_clean['drug_name'].value_counts().head(5))


drug_name
Sulfotanshinone Sodium Injection    1
Furosemide Tablet                   1
Metoprolol Tartrate Injection       1
Aspirin Enteric-Coated Tablet       1
Name: count, dtype: int64


Cleaned Data verification for patient_prescriptions

In [107]:
cardiac.to_csv(
    CLEANED_PATH / "patient_prescriptions_cleaned.csv",
    index=False
)

print("Saved:", CLEANED_PATH / "patient_prescriptions_cleaned.csv")

Saved: cleaned_data/patient_prescriptions_cleaned.csv


Saved:/Users/apple/Desktop/Numpy_Ninja/Python_Session/Python_Hackathon_Sep_2026/cleaned_data/cardiac_failure\cardiac_cleaned.csv

In [108]:
patient_prescriptions_check = pd.read_csv("cleaned_data/patient_prescriptions_cleaned.csv")
  

print("Saved shape:", patient_prescriptions_check.shape)
print("Saved columns:", patient_prescriptions_check.columns.tolist())

Saved shape: (2008, 14)
Saved columns: ['inpatient_number', 'nyha_cardiac_function_classification', 'killip_grade', 'myocardial_infarction', 'congestive_heart_failure', 'peripheral_vascular_disease', 'type_of_heart_failure', 'lvef', 'left_ventricular_end_diastolic_diameter_lv', 'mitral_valve_ems', 'mitral_valve_ams', 'ea', 'tricuspid_valve_return_velocity', 'tricuspid_valve_return_pressure']


In [109]:
dataset_overview = pd.DataFrame([
    {
        "Dataset": name,
        "Rows": df.shape[0],
        "Columns": df.shape[1],
        "Unique Patients": df["inpatient_number"].nunique(),
        "Duplicate Patient IDs": df["inpatient_number"].duplicated().sum(),
        "One Row per Patient": df.shape[0] == df["inpatient_number"].nunique()
    }
    for name, df in datasets.items()
])

dataset_overview

,Dataset,Rows,Columns,Unique Patients,Duplicate Patient IDs,One Row per Patient
0,Demography,2009,7,2009,0,True
1,Hospitalization & Discharge,2008,21,2008,0,True
2,Cardiac Complications,2008,14,2008,0,True
3,Labs,2008,107,2008,0,True
4,Patient Prescriptions,15362,2,2007,13355,False
5,Patient History,2008,17,2008,0,True
6,Responsiveness,2008,6,2008,0,True
